# FOMC statements -- monthly update

Re-scrapes the document links (cheap -- it's just link discovery), finds
meeting dates not already in the clean dataset, fetches and cleans just
those statements, and appends.

In [1]:
import os
import pandas as pd
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from fomc_helpers import (
    scrape_all_fomc_docs, scrape_calendar, concat_historical_and_calendar,
    filter_statement_html, fetch_statement_texts, clean_fomc_statement, url_doc_letter,
)

BASE = os.path.dirname(os.getcwd())
RAW_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fomc_docs_raw.csv")
COMPILED_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fomc_statements_compiled_raw.csv")
CLEAN_PATH = os.path.join(BASE, "Data Output", "fomc_statements_clean.csv")

## Where are we

In [2]:
existing = pd.read_csv(CLEAN_PATH, parse_dates=["date"])
last_date = existing["date"].max()
print(f"Last statement on file: {last_date.date()}")

Last statement on file: 2026-09-16


## Re-scrape links and find new meeting dates

In [3]:
df_hist = scrape_all_fomc_docs(pause=0.4, verbose=False)
df_cal = scrape_calendar()
df_combined = concat_historical_and_calendar(df_hist, df_cal)
df_combined.to_csv(RAW_PATH, index=False)

df_statements_html = filter_statement_html(df_combined)
new_rows = df_statements_html[df_statements_html["date"] > last_date]
print(f"{len(new_rows)} new statement link(s) found")

0 new statement link(s) found


## Fetch, clean, and append anything new

In [4]:
if len(new_rows):
    fresh = fetch_statement_texts(new_rows, pause=0.2)

    compiled_existing = pd.read_csv(COMPILED_PATH, parse_dates=["date"])
    fresh_compiled = fresh.rename(columns={"text": "raw_text"})[
        ["date", "year", "url", "doc_type", "page_title", "raw_text", "word_count", "http_status", "error"]
    ]
    compiled_updated = pd.concat([compiled_existing, fresh_compiled], ignore_index=True).drop_duplicates(subset=["url"]).sort_values("date").reset_index(drop=True)
    compiled_updated.to_csv(COMPILED_PATH, index=False)

    fresh["clean_text"] = fresh["text"].map(lambda s: clean_fomc_statement(s, min_chars=30, flatten=False))
    fresh["doc_letter"] = fresh["url"].map(url_doc_letter)
    fresh["is_main_statement"] = fresh["doc_letter"] == "a"
    fresh["title"] = fresh["page_title"].str.replace(r"^Federal Reserve Board\s*-\s*", "", regex=True)
    fresh_out = pd.DataFrame({
        "date": fresh["date"], "year": fresh["year"], "speaker": None,
        "doc_type": "FOMC_statement", "title": fresh["title"],
        "is_main_statement": fresh["is_main_statement"], "source_corpus": "fomc_statements",
        "url": fresh["url"], "word_count": fresh["clean_text"].str.split().str.len(),
        "clean_text": fresh["clean_text"], "raw_text": fresh["text"],
    })
    updated = pd.concat([existing, fresh_out], ignore_index=True).drop_duplicates(subset=["url"]).sort_values("date").reset_index(drop=True)
    updated.to_csv(CLEAN_PATH, index=False)
    print(f"Appended {len(fresh_out)} row(s), {len(updated):,} total")
else:
    print("Nothing new.")

Nothing new.
